# 02c Estimate: the regression models plus the route indicator

In `01b_preprocess_indicators.ipynb`, we built an indicator from Frontex detections on the routes towards Switzerland, hoping that movements on the routes could announce changes in Swiss applications. In the present notebook, we test this idea: we add the indicator to the three regression models of `02b_estimate_regression.ipynb` and run them in exactly the same backtest.

Everything else stays the same: the target, the other features, the models, the penalties and the forecast origins. This follows the rule of our project plan, one block at a time. If we changed several things at once, we could not tell what caused a gain or a loss; by changing only the indicator, any difference in the errors can be attributed to it.

We read the clean target from `01_preprocess.ipynb` (`../data/processed/asylum_clean.csv`) and the clean indicator from `01b_preprocess_indicators.ipynb` (`../data/processed/routes_clean.csv`). We save the forecasts in `../data/processed/backtest_forecasts_indicator.csv`, with the same columns as `backtest_forecasts_regression.csv`.

As before, the test period (October 2024 to September 2026) is **not used**, so that it stays a fair check at the end. We compare the models in `03_evaluate.ipynb`.

In [1]:
import numpy as np
import pandas as pd
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

np.random.seed(42)

## 1. Load the target and the indicator

We put the log of the indicator next to the target, month by month, so that each row holds both series for the same month. The indicator starts in January 2009 and the target in January 2008, so the first 12 months of the indicator are missing (`NaN`). `reindex` puts the indicator on the months of the target, so that both series use the same row positions.

*Not seen in class:* `reindex(months)` puts a series on a given list of row labels and leaves `NaN` where a label is missing.

In [2]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")

routes = pd.read_csv("../data/processed/routes_clean.csv")
routes["month"] = pd.to_datetime(routes["month"])
routes = routes.set_index("month")

y = clean["applicants_total"].to_numpy(dtype=float)
log_y = clean["log_applicants"].to_numpy(dtype=float)
months = clean.index
log_routes = routes["log_routes"].reindex(months).to_numpy(dtype=float)   # NaN before 2009-01

FIRST_ORIGIN = pd.Timestamp("2012-12-01")
LAST_BACKTEST = pd.Timestamp("2024-09-01")
H_MAX = 3
first_origin = int(np.where(months == FIRST_ORIGIN)[0][0])
last_backtest = int(np.where(months == LAST_BACKTEST)[0][0])

print("Indicator missing for", np.isnan(log_routes).sum(), "months (before", routes.index[0], ")")

Indicator missing for 12 months (before 2009-01-01 00:00:00 )


## 2. The two indicator features

The most important rule here is timing, fixed in `01b_preprocess_indicators.ipynb`: **at a forecast origin $t$, the newest indicator month we may use is $t-1$**, because Frontex is published one month later than the target. Using month $t$ would give the model information a planner did not yet have.

We decided on two features before running any model, so that we could not pick the ones that happen to work best. With $R$ = the detections on the three routes, they are:

| Feature | Formula | Idea |
|---|---|---|
| `routes_change_last_month` | $\log R_{t-1} - \log R_{t-2}$ | the newest monthly change on the routes |
| `routes_change_3_months` | $\log R_{t-1} - \log R_{t-4}$ | the change over the last three known months |

They are **changes**, like the other features, because trending levels can look related without any real link. We did not use a change over a whole year: it would need the indicator 13 months back, and the first training windows would become very small (in the first validation block, only about 6 rows for 16 features), too small to fit the model.

All other features, the target, the models, the grids of `alpha` and the validation inside each training window are **exactly those of `02b_estimate_regression.ipynb`**, so that the indicator is the only difference.

In [3]:
def make_table(h):
    # Same table as in 02b, plus the two indicator features (indicator months up to t - 1 only)
    rows = []
    for t in range(12, len(log_y) - h):
        if np.isnan(log_routes[t - 4]):           # the indicator is needed back to month t - 4
            continue
        rows.append({
            "t": t,
            "change_last_month": log_y[t] - log_y[t - 1],
            "change_month_before": log_y[t - 1] - log_y[t - 2],
            "same_change_last_year": log_y[t + h - 12] - log_y[t - 12],
            "routes_change_last_month": log_routes[t - 1] - log_routes[t - 2],
            "routes_change_3_months": log_routes[t - 1] - log_routes[t - 4],
            "calendar_month": months[t + h].month,
            "target": log_y[t + h] - log_y[t],
        })
    table = pd.DataFrame(rows).set_index("t")
    dummies = pd.get_dummies(table["calendar_month"], prefix="month", drop_first=True).astype(float)
    features = ["change_last_month", "change_month_before", "same_change_last_year",
                "routes_change_last_month", "routes_change_3_months"]
    X = pd.concat([table[features], dummies], axis=1)
    return X, table["target"]

tables = {h: make_table(h) for h in range(1, H_MAX + 1)}

X2, target2 = tables[2]
print("Table for h = 2:", X2.shape[0], "rows and", X2.shape[1], "features")
print("First row: origin", months[X2.index.min()], "| missing values:", X2.isna().sum().sum())
X2[["routes_change_last_month", "routes_change_3_months"]].describe().round(3)

Table for h = 2: 207 rows and 16 features
First row: origin 2009-05-01 00:00:00 | missing values: 0


,routes_change_last_month,routes_change_3_months
count,207.000,207.000
mean,0.002,0.006
std,0.376,0.716
min,-1.645,-2.216
25%,-0.229,-0.475
50%,0.039,0.093
75%,0.218,0.479
max,1.227,2.384


As we can see, for $h = 2$ the table has 207 rows and 16 features: the three own changes, the two route changes and 11 month dummies. No value is missing. The first row is the origin May 2009, because the 3-month route change needs the indicator back to $t - 4$ and the indicator starts in January 2009. In `02b` the table starts in January 2009, so the indicator models have 4 fewer training rows.

The two route features are centred near 0 (means 0.002 and 0.006), but they move a lot: the standard deviation is 0.38 for the monthly change and 0.72 for the 3-month change, and the largest monthly fall is −1.65 in logs.

Note also that the route numbers jump much more from month to month than the applications in Switzerland. Such a noisy feature can help a model only if its signal is strong; otherwise it mostly adds noise. The backtest will tell.

## 3. Models, grids and the choice of `alpha`: the same as in `02b`

The functions below are copied from `02b_estimate_regression.ipynb` without any change: scaling inside the pipeline, two validation blocks of 12 months inside each training window, and the same grids of `alpha`. Copying them exactly is what makes the comparison fair: if we tuned these models differently, a difference in the errors could come from the tuning and not from the indicator.

In [4]:
def make_model(name, alpha=None):
    if name == "Linear regression + routes":
        return make_pipeline(StandardScaler(), LinearRegression())
    if name == "Ridge + routes":
        return make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    return make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))

alpha_grids = {
    "Linear regression + routes": [None],
    "Ridge + routes": [0.1, 1, 10, 100, 1000, 10000],
    "Lasso + routes": [0.0001, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3],
}

def choose_alpha(name, X, target, T, h, n_blocks=2, block=12):
    grid = alpha_grids[name]
    if len(grid) == 1:
        return grid[0]
    first_row, last_row = X.index.min(), T - h
    scores = []
    for alpha in grid:
        mse_blocks = []
        for b in range(n_blocks):
            start = last_row - n_blocks * block + 1 + b * block
            valid_rows = list(range(start, start + block))
            train_rows = list(range(first_row, start - h + 1))
            model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
            errors = target.loc[valid_rows].to_numpy() - model.predict(X.loc[valid_rows])
            mse_blocks.append(np.mean(errors ** 2))
        scores.append(np.mean(mse_blocks))
    return grid[int(np.argmin(scores))]

## 4. The backtest loop

The loop is the same as in `02b`, with the same 141 origins and the same horizons, so that the models with and without the indicator are scored on exactly the same months. Inside the loop, an `assert` checks again that no training target lies after the origin: if one ever did, the code would stop instead of producing a forecast that uses its own future.

In [5]:
rows = []

for t in range(first_origin, last_backtest):
    for h in range(1, H_MAX + 1):
        target_pos = t + h
        if target_pos > last_backtest:
            continue
        X, target = tables[h]
        train_rows = list(range(X.index.min(), t - h + 1))
        assert max(train_rows) + h <= t                    # no training target after the origin

        for name in alpha_grids:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                alpha = choose_alpha(name, X, target, t, h)
                model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
            forecast_log = log_y[t] + model.predict(X.loc[[t]])[0]
            rows.append({
                "model": name, "origin": months[t], "month": months[target_pos], "h": h,
                "forecast_log": forecast_log, "actual_log": log_y[target_pos],
                "forecast": np.exp(forecast_log), "actual": y[target_pos], "alpha": alpha,
            })

forecasts = pd.DataFrame(rows)
print("Number of stored forecasts:", forecasts.shape[0])

Number of stored forecasts: 1260


## 5. Checks on the stored forecasts

As in `02_estimate.ipynb` and `02b`, we check the result directly: the number of forecasts per model and horizon, that no forecast month lies in the test period, that every forecast month lies after its origin, and that no forecast is missing.

In [6]:
print(forecasts.groupby(["model", "h"]).size())
print()
print("All forecast months in the backtest (before 2024-10):", (forecasts["month"] < pd.Timestamp("2024-10-01")).all())
print("All forecast months after their origin:", (forecasts["month"] > forecasts["origin"]).all())
print("Missing or infinite forecasts:", (~np.isfinite(forecasts["forecast_log"])).sum())

model                       h
Lasso + routes              1    141
                            2    140
                            3    139
Linear regression + routes  1    141
                            2    140
                            3    139
Ridge + routes              1    141
                            2    140
                            3    139
dtype: int64

All forecast months in the backtest (before 2024-10): True
All forecast months after their origin: True
Missing or infinite forecasts: 0


The checks show that, as in `02b`, each model has 141, 140 and 139 forecasts for $h = 1$, 2 and 3 (1,260 together), all in the backtest, all after their origin, none missing.

This matters because, together with the `assert` in the loop, it shows that the indicator models used only data known at each origin and never touched the test period. They can be compared fairly with the same models without the indicator.

### Which penalties were chosen?

As in `02b`, we check that the chosen `alpha` does not sit mostly at an edge of the grid, because that would mean the best value may lie outside it.

In [7]:
for name in ["Ridge + routes", "Lasso + routes"]:
    chosen = forecasts.loc[forecasts["model"] == name, "alpha"]
    print(f"{name}: how often each alpha was chosen (grid from {alpha_grids[name][0]} to {alpha_grids[name][-1]})")
    print(chosen.value_counts().sort_index().to_string())
    print()

Ridge + routes: how often each alpha was chosen (grid from 0.1 to 10000)
alpha
0.1         92
1.0          9
10.0        50
100.0      196
1000.0      65
10000.0      8

Lasso + routes: how often each alpha was chosen (grid from 0.0001 to 0.3)
alpha
0.0001    100
0.0010     11
0.0030     22
0.0100     93
0.0300     78
0.1000     66
0.3000     50



Looking at the chosen penalties, **Ridge + routes** most often chose `alpha = 100` (196 of 420 times). The largest value (10,000) was chosen 8 times, the smallest (0.1) 92 times. **Lasso + routes** most often chose 0.0001 (100 times), then 0.01 (93 times) and 0.03 (78 times). The largest value (0.3) was chosen 50 times.

This tells us that the upper edges are rarely reached. The lower edge means almost no penalty, which is the linear regression we already run. This is the same pattern as in `02b`, so we keep the same grids. Keeping them also keeps the comparison fair: only the indicator block changes.

## 6. What do the models do with the indicator?

Before measuring the errors, we look at whether the models use the indicator at all. As in `02b`, we look at the coefficients for $h = 2$ at the last origin (August 2024), on standardised features, so that they can be compared with each other. We show only the five non-calendar features, because the question is about the routes. Again, a coefficient says what helps to *predict*, not what *causes*.

In [8]:
t = last_backtest - 1
h = 2
X, target = tables[h]
train_rows = list(range(X.index.min(), t - h + 1))

coefficients = pd.DataFrame(index=X.columns)
for name in ["Ridge + routes", "Lasso + routes"]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha(name, X, target, t, h)
        model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
    coefficients[f"{name} (alpha = {alpha})"] = model[-1].coef_     # the last step of the pipeline is the model

print("Origin:", months[t], "| horizon h =", h, "| training rows:", len(train_rows))
coefficients.iloc[:5].round(3)

Origin: 2024-08-01 00:00:00 | horizon h = 2 | training rows: 182


,Ridge + routes (alpha = 0.1),Lasso + routes (alpha = 0.0001)
change_last_month,-0.016,-0.015
change_month_before,-0.016,-0.016
same_change_last_year,-0.001,-0.001
routes_change_last_month,0.021,0.021
routes_change_3_months,-0.053,-0.053


As we can see, at the last origin (182 training rows) Ridge and Lasso chose their smallest penalty and give almost the same coefficients. Among these five features, the 3-month route change has the largest coefficient in size (−0.053), and the monthly route change has +0.021. The two own recent changes are at about −0.016 each, and the change one year earlier at −0.001.

So the models do use the indicator: its coefficients are not zero. But the two route features have opposite signs, and the 3-month change contains the last monthly change (both use $\log R_{t-1}$), so the two partly cancel each other, and we should not read each sign on its own. A model can use a feature without forecasting better because of it. The real question is whether the indicator makes the *forecasts* better, which we check in `03_evaluate.ipynb`.

## 7. Save the forecasts

We save all forecasts, so that `03_evaluate.ipynb` can compare each model with its own version without the indicator, on exactly the same months.

In [9]:
forecasts.to_csv("../data/processed/backtest_forecasts_indicator.csv", index=False)
print("Saved", forecasts.shape[0], "forecasts with the columns", list(forecasts.columns))

Saved 1260 forecasts with the columns ['model', 'origin', 'month', 'h', 'forecast_log', 'actual_log', 'forecast', 'actual', 'alpha']


## Summary

**The regressions of `02b`, plus the route indicator.** We keep the linear regression, Ridge and Lasso with the same features, target, grids and validation, and add two route features: the change of log detections in the last known month and over the last three known months. Changing only this block lets us attribute any difference in the errors to the indicator.

**A strict timing rule.** At origin $t$ we use the indicator only up to month $t - 1$, because Frontex is published one month later than the target (`01b_preprocess_indicators.ipynb`).

**The same honest rules as before.** The same 141 origins and horizons; scaling and the choice of `alpha` happen inside each training window; an `assert` checks that no training target lies after the origin; the test period is not used.

**The models use the indicator, but whether it helps is still open.** The route coefficients are not zero, but they partly cancel each other. `03_evaluate.ipynb` compares the forecasts with the same models without the indicator.

*Not seen in class (marked above):* `reindex(months)`.

The 1,260 forecasts are saved in `../data/processed/backtest_forecasts_indicator.csv`.